# 02. FMA 원곡 매칭

정제한 TTA의 `original_audio` 296개를 FMA 메타데이터와 연결한다. 후보가 여러 곡이면 라이선스·장르·track ID 순으로 고른다. 여기서는 파일을 내려받기 전, 필요한 REAL 곡의 ID를 확정한다.

## 0. 프로젝트 경로 설정

노트북을 `project/` 또는 `project/notebooks/`에서 실행해도 프로젝트 루트를 자동으로 찾도록 한다.

In [17]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import unicodedata

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    if (PROJECT_ROOT.parent / "data").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

ECHOES_MANIFEST = PROJECT_ROOT / "data/raw/Echoes/Echoes/dataset_manifest.csv"
FMA_TRACKS = PROJECT_ROOT / "data/raw/FMA/fma_metadata/tracks.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Echoes manifest exists:", ECHOES_MANIFEST.exists())
print("FMA tracks.csv exists:", FMA_TRACKS.exists())

PROJECT_ROOT: /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project
Echoes manifest exists: True
FMA tracks.csv exists: True


## 1. Echoes Clean TTA 다시 생성

앞 노트북과 같은 정제 규칙을 적용한다.

1. `type == "TTA"`만 사용
2. 동일 `path_in_dataset`을 여러 행이 공유하는 경우 모두 제외
3. 남은 `original_audio` 고유값을 추출

In [18]:
echoes = pd.read_csv(ECHOES_MANIFEST)

tta = echoes[echoes["type"] == "TTA"].copy()

dup_mask = tta["path_in_dataset"].duplicated(keep=False)
tta_clean = tta[~dup_mask].copy()

originals = (
    tta_clean[["original_audio", "genre"]]
    .drop_duplicates("original_audio")
    .sort_values("original_audio")
    .reset_index(drop=True)
)

print("Original TTA rows :", len(tta))
print("Excluded rows     :", int(dup_mask.sum()))
print("Clean TTA rows    :", len(tta_clean))
print("Original groups   :", len(originals))

display(originals.head(10))

Original TTA rows : 3165
Excluded rows     : 3
Clean TTA rows    : 3162
Original groups   : 296


,original_audio,genre
0,"10,000 People Chanting, ""I'm an Individual"" - ...",Electronic
1,1984 - Punk Rock Opera,Rock
2,2 (Wasn't There) - Isle of Pine,Rock
3,2Much (Andy Spinelli & Alex Sánchez House Edit...,Electronic
4,3 am West End - statusq,Electronic
5,5 (Lexington) - Isle of Pine,Rock
6,"50,000 Volts of Democracy mp3 - Legally Blind",Rock
7,"6 (Coat of Arms, Close) - Isle of Pine",Rock
8,A Dark Blue Arc - Pipe Choir,Rock
9,A Different World By Night - Nihilore,Electronic


### 기대 결과

```text
Original TTA rows : 3165
Excluded rows     : 3
Clean TTA rows    : 3162
Original groups   : 296
```

## 2. FMA tracks.csv 로드

FMA `tracks.csv`는 MultiIndex 컬럼을 사용한다. 매칭에 필요한 정보만 별도 DataFrame으로 만든다.

- `track_id`
- title
- artist
- genre_top
- license
- duration
- subset

In [19]:
fma = pd.read_csv(FMA_TRACKS, header=[0, 1], index_col=0)

fma_simple = pd.DataFrame(
    {
        "track_id": fma.index.astype(int),
        "title": fma[("track", "title")].values,
        "artist": fma[("artist", "name")].values,
        "genre_top": fma[("track", "genre_top")].values,
        "license": fma[("track", "license")].values,
        "duration": fma[("track", "duration")].values,
        "subset": fma[("set", "subset")].values,
    }
)

print("FMA tracks:", len(fma_simple))
display(fma_simple.head())

FMA tracks: 106574


,track_id,title,artist,genre_top,license,duration,subset
0,2,Food,AWOL,Hip-Hop,Attribution-NonCommercial-ShareAlike 3.0 Inter...,168,small
1,3,Electric Ave,AWOL,Hip-Hop,Attribution-NonCommercial-ShareAlike 3.0 Inter...,237,medium
2,5,This World,AWOL,Hip-Hop,Attribution-NonCommercial-ShareAlike 3.0 Inter...,206,small
3,10,Freeway,Kurt Vile,Pop,Attribution-NonCommercial-NoDerivatives (aka M...,161,small
4,20,Spiritual Level,Nicky Cook,NaN,Attribution-NonCommercial-NoDerivatives (aka M...,311,large


## 3. 문자열 정규화

Echoes의 `original_audio`는 보통 `곡 제목 - 아티스트` 형식이다.

FMA에서도 `track.title + " - " + artist.name`을 만들어 비교한다.

대소문자, Unicode 표현, 앞뒤 공백, 중복 공백만 정규화하고, 우선은 보수적인 exact matching을 수행한다.

In [20]:
# 문자열 정규화
def normalize_text(x):
    if pd.isna(x):
        return ""
    x = unicodedata.normalize("NFKC", str(x))
    x = x.strip().lower()
    x = re.sub(r"\s+", " ", x)
    return x


fma_simple["fma_name"] = (
    fma_simple["title"].fillna("").astype(str).str.strip()
    + " - "
    + fma_simple["artist"].fillna("").astype(str).str.strip()
)

fma_simple["match_key"] = fma_simple["fma_name"].map(normalize_text)
originals["match_key"] = originals["original_audio"].map(normalize_text)

display(fma_simple[["track_id", "title", "artist", "fma_name", "match_key"]].head())

,track_id,title,artist,fma_name,match_key
0,2,Food,AWOL,Food - AWOL,food - awol
1,3,Electric Ave,AWOL,Electric Ave - AWOL,electric ave - awol
2,5,This World,AWOL,This World - AWOL,this world - awol
3,10,Freeway,Kurt Vile,Freeway - Kurt Vile,freeway - kurt vile
4,20,Spiritual Level,Nicky Cook,Spiritual Level - Nicky Cook,spiritual level - nicky cook


## 4. Exact matching 후보 개수 확인

각 Echoes `original_audio`에 대해 FMA에서 동일한 `match_key`가 몇 개 존재하는지 센다.

- **1개 후보**: 거의 바로 연결 가능
- **2개 이상 후보**: 추가 규칙 필요
- **0개 후보**: 문자열 차이 또는 metadata 불일치 조사 필요

In [21]:
# Exact matching 후보 개수 확인
candidate_counts = fma_simple.groupby("match_key").size().rename("candidate_count")

match_summary = originals.merge(
    candidate_counts, left_on="match_key", right_index=True, how="left"
)

match_summary["candidate_count"] = (
    match_summary["candidate_count"].fillna(0).astype(int)
)

print("===== EXACT MATCH SUMMARY =====")
print(match_summary["candidate_count"].value_counts().sort_index())

print("\nExactly 1 candidate :", int((match_summary["candidate_count"] == 1).sum()))
print("Multiple candidates :", int((match_summary["candidate_count"] > 1).sum()))
print("No candidate        :", int((match_summary["candidate_count"] == 0).sum()))
print("Total               :", len(match_summary))

===== EXACT MATCH SUMMARY =====
candidate_count
1    279
2     16
5      1
Name: count, dtype: int64

Exactly 1 candidate : 279
Multiple candidates : 17
No candidate        : 0
Total               : 296


## Exact Matching 결과 확인

Echoes의 Clean TTA 데이터에는 총 **296개의 고유 `original_audio`**가 존재한다.

각 `original_audio`에 대해 FMA의 `track title + artist name` 조합과 정확히 일치하는 후보가 몇 개 존재하는지 확인하였다.

매칭 결과는 다음 세 가지로 구분하였다.

- **Exactly 1 candidate**: FMA에서 정확히 하나의 곡만 검색된 경우
- **Multiple candidates**: 동일한 제목과 아티스트를 가진 FMA track이 두 개 이상 존재하는 경우
- **No candidate**: FMA에서 일치하는 곡을 찾지 못한 경우

확인 결과 총 296개의 `original_audio` 중 279개는 하나의 후보와 정확히 일치하였으며, 17개는 두 개 이상의 후보가 존재하였다. 매칭되지 않은 원곡은 없었다.

따라서 모든 `original_audio`를 FMA metadata에서 찾을 수 있었으며, 이후에는 17개의 복수 후보에 대해서만 추가 검토가 필요하다.

## 5. 매칭되지 않은 원곡 확인

In [22]:
# 매칭되지 않은 원곡 확인
no_match = match_summary[match_summary["candidate_count"] == 0].copy()

print("No-match count:", len(no_match))
display(no_match[["original_audio", "genre"]])

No-match count: 0


,original_audio,genre


## 6. 복수 후보 원곡 확인

같은 `곡 제목 - 아티스트`가 FMA에 여러 번 존재할 수 있다. 이 경우 자동으로 첫 번째 곡을 선택하면 안 된다.

후보들의 `track_id`, `genre_top`, `license`, `duration`, `subset`을 함께 확인한다.

In [23]:
# 복수 후보 원곡 확인
multiple = match_summary[match_summary["candidate_count"] > 1].copy()

print("Multiple-match original_audio count:", len(multiple))
display(multiple[["original_audio", "genre", "candidate_count"]])

Multiple-match original_audio count: 17


,original_audio,genre,candidate_count
1,1984 - Punk Rock Opera,Rock,2
18,"Aquamarine, My Distant Blue - Nihilore",Electronic,2
20,As Nihilism Gives Way To Existentialism - Nihi...,Electronic,2
96,I Know His Blood - Vienna Ditto,Electronic,2
98,I'm gonna try to reach - Los Llamarada,Rock,2
110,KOMFORT - voyageurs,Rock,2
122,Let You're Body Move - D SMILEZ,Electronic,2
130,Lost In The Music (D-Smilez Mix) - D SMILEZ,Electronic,2
133,Loved Ones - Rowan Box,Electronic,2
140,Monkeystage - Ergo Phizmiz,Pop,2


## FMA 복수 후보 상세 확인

Exact Matching 결과, **17개의 `original_audio`가 두 개 이상의 FMA track과 일치**하였다.

이는 FMA 데이터 안에 동일한 곡 제목과 아티스트명을 가지면서 서로 다른 `track_id`로 등록된 음악이 존재하기 때문이다.

따라서 단순히 첫 번째 검색 결과를 선택하지 않고, 각 후보의 다음 정보를 비교한다.

- `track_id`
- 곡 제목
- 아티스트
- `genre_top`
- 라이선스
- 재생시간
- FMA subset

이 단계의 목적은 복수 후보의 특성을 확인하고, 이후 최종 REAL 음악을 선택하기 위한 **일관되고 재현 가능한 선택 규칙**을 설정하는 것이다.

In [24]:
# FMA 복수 후보 상세 확인
multi_candidates = (
    multiple[["original_audio", "genre", "match_key"]]
    .merge(
        fma_simple[
            [
                "track_id",
                "title",
                "artist",
                "genre_top",
                "license",
                "duration",
                "subset",
                "fma_name",
                "match_key",
            ]
        ],
        on="match_key",
        how="left",
    )
    .sort_values(["original_audio", "track_id"])
)

display(
    multi_candidates
)  # Echoes와 FMA 매칭 중 17개의 original_audio가 2개 이상이 후보의 음악을 추출함

,original_audio,genre,match_key,track_id,title,artist,genre_top,license,duration,subset,fma_name
0,1984 - Punk Rock Opera,Rock,1984 - punk rock opera,137212,1984,Punk Rock Opera,Rock,Attribution-NonCommercial,203,small,1984 - Punk Rock Opera
1,1984 - Punk Rock Opera,Rock,1984 - punk rock opera,149410,1984,Punk Rock Opera,Rock,Attribution,200,medium,1984 - Punk Rock Opera
2,"Aquamarine, My Distant Blue - Nihilore",Electronic,"aquamarine, my distant blue - nihilore",134150,"Aquamarine, My Distant Blue",Nihilore,Electronic,Creative Commons Attribution,141,large,"Aquamarine, My Distant Blue - Nihilore"
3,"Aquamarine, My Distant Blue - Nihilore",Electronic,"aquamarine, my distant blue - nihilore",140002,"Aquamarine, My Distant Blue",Nihilore,Electronic,Attribution,141,medium,"Aquamarine, My Distant Blue - Nihilore"
4,As Nihilism Gives Way To Existentialism - Nihi...,Electronic,as nihilism gives way to existentialism - nihi...,134148,As Nihilism Gives Way To Existentialism,Nihilore,Electronic,Creative Commons Attribution,335,large,As Nihilism Gives Way To Existentialism - Nihi...
5,As Nihilism Gives Way To Existentialism - Nihi...,Electronic,as nihilism gives way to existentialism - nihi...,140000,As Nihilism Gives Way To Existentialism,Nihilore,Electronic,Attribution,335,medium,As Nihilism Gives Way To Existentialism - Nihi...
6,I Know His Blood - Vienna Ditto,Electronic,i know his blood - vienna ditto,107616,I Know His Blood,Vienna Ditto,Electronic,Attribution,238,small,I Know His Blood - Vienna Ditto
7,I Know His Blood - Vienna Ditto,Electronic,i know his blood - vienna ditto,147298,I Know His Blood,Vienna Ditto,Electronic,Attribution,238,medium,I Know His Blood - Vienna Ditto
8,I'm gonna try to reach - Los Llamarada,Rock,i'm gonna try to reach - los llamarada,28763,I'm gonna try to reach,Los Llamarada,Rock,Attribution-NoDerivatives 3.0 International,98,large,I'm gonna try to reach - Los Llamarada
9,I'm gonna try to reach - Los Llamarada,Rock,i'm gonna try to reach - los llamarada,84389,I'm Gonna Try To Reach,Los Llamarada,Rock,Attribution-Noncommercial-Share Alike 3.0 Unit...,98,large,I'm Gonna Try To Reach - Los Llamarada


## 7. 정확히 1개 후보인 원곡의 매칭 결과 확인

In [25]:
# 정확히 1개 후보인 원곡의 매칭 결과 확인
single = match_summary[match_summary["candidate_count"] == 1][
    ["original_audio", "genre", "match_key"]
].copy()

single_matches = single.merge(
    fma_simple[
        [
            "track_id",
            "title",
            "artist",
            "genre_top",
            "license",
            "duration",
            "subset",
            "fma_name",
            "match_key",
        ]
    ],
    on="match_key",
    how="left",
)

print("Single exact matches:", len(single_matches))
display(single_matches.head(20))

Single exact matches: 279


,original_audio,genre,match_key,track_id,title,artist,genre_top,license,duration,subset,fma_name
0,"10,000 People Chanting, ""I'm an Individual"" - ...",Electronic,"10,000 people chanting, ""i'm an individual"" - ...",140932,"10,000 People Chanting, ""I'm an Individual""",Nihilore,Electronic,Creative Commons Attribution,372,medium,"10,000 People Chanting, ""I'm an Individual"" - ..."
1,2 (Wasn't There) - Isle of Pine,Rock,2 (wasn't there) - isle of pine,66449,2 (Wasn't There),Isle of Pine,Rock,Attribution-NoDerivs 2.5 Canada,108,medium,2 (Wasn't There) - Isle of Pine
2,2Much (Andy Spinelli & Alex Sánchez House Edit...,Electronic,2much (andy spinelli & alex sánchez house edit...,114244,2Much (Andy Spinelli & Alex Sánchez House Edit),Tentacles,Electronic,Attribution,486,medium,2Much (Andy Spinelli & Alex Sánchez House Edit...
3,3 am West End - statusq,Electronic,3 am west end - statusq,112378,3 am West End,statusq,Electronic,Attribution,291,medium,3 am West End - statusq
4,5 (Lexington) - Isle of Pine,Rock,5 (lexington) - isle of pine,66445,5 (Lexington),Isle of Pine,Rock,Attribution-NoDerivs 2.5 Canada,157,large,5 (Lexington) - Isle of Pine
5,"50,000 Volts of Democracy mp3 - Legally Blind",Rock,"50,000 volts of democracy mp3 - legally blind",130401,"50,000 Volts of Democracy mp3",Legally Blind,Rock,Attribution,270,medium,"50,000 Volts of Democracy mp3 - Legally Blind"
6,"6 (Coat of Arms, Close) - Isle of Pine",Rock,"6 (coat of arms, close) - isle of pine",66446,"6 (Coat of Arms, Close)",Isle of Pine,Rock,Attribution-NoDerivs 2.5 Canada,204,large,"6 (Coat of Arms, Close) - Isle of Pine"
7,A Dark Blue Arc - Pipe Choir,Rock,a dark blue arc - pipe choir,129963,A Dark Blue Arc,Pipe Choir,Rock,Attribution,327,medium,A Dark Blue Arc - Pipe Choir
8,A Different World By Night - Nihilore,Electronic,a different world by night - nihilore,140926,A Different World By Night,Nihilore,Electronic,Creative Commons Attribution,296,small,A Different World By Night - Nihilore
9,A Lady In Red With A Plan To Steal - Did You J...,Rock,a lady in red with a plan to steal - did you j...,74910,A Lady In Red With A Plan To Steal,Did You Just Hex Me?,Rock,Creative Commons Attribution,134,medium,A Lady In Red With A Plan To Steal - Did You J...


## 8. 장르 일치 여부 간단 점검

Echoes genre와 FMA `genre_top`의 일치 여부를 확인한다.

장르가 다르다고 바로 제거하지는 않는다. 두 데이터셋의 장르 분류 기준이 완전히 같다고 보장할 수 없으므로 품질 점검용으로만 사용한다.

In [26]:
# 장르 일치 여부 간단 점검
single_matches["genre_match"] = (
    single_matches["genre"].astype(str).str.lower()
    == single_matches["genre_top"].astype(str).str.lower()
)

print(single_matches["genre_match"].value_counts(dropna=False))

display(
    single_matches.loc[
        ~single_matches["genre_match"],
        ["original_audio", "genre", "track_id", "genre_top", "title", "artist"],
    ].head(30)
)

genre_match
True    279
Name: count, dtype: int64


,original_audio,genre,track_id,genre_top,title,artist


## 9. Exact matching 핵심 결과

```text
Exactly 1 candidate : 279
Multiple candidates : 17
No candidate        : 0
```

세 숫자의 합은 **296**이어야 한다.

이 결과를 확인한 뒤:
1. 복수 후보 선택 규칙 확정
2. 무매칭 원곡 보정
3. 최종 `original_audio → FMA track_id` 매핑 확정

으로 진행한다.

---

## 10. 이후 개발 파일

매칭 규칙이 확정되면 실제 개발 스크립트를 별도로 만든다.

예정:
```text
src/01_build_master_manifest.py
```

지금은 아직 후보 검증 단계이므로 자동 선택 로직을 먼저 확정하지 않는다.

## 10. Echoes 전체 original_audio 개수 확인

Clean TTA 데이터에서 고유한 `original_audio`가 296개임을 확인하였다.

이 296개가 MusicGen 중복 데이터 3개를 제거하면서 감소한 결과인지, 아니면 Echoes manifest 자체가 처음부터 296개의 고유 `original_audio`를 가지고 있는지 확인하기 위해 전체 데이터를 비교하였다.

다음 항목의 고유 `original_audio` 개수를 확인하였다.

- 전체 Echoes
- TTA
- ATA
- Clean TTA

확인 결과 네 경우 모두 **296개**로 동일하였다.

따라서 296이라는 값은 중복 데이터를 제거하면서 만들어진 값이 아니라, 현재 Echoes manifest에 존재하는 **고유한 `original_audio` 이름의 수**이다.

In [27]:
# Echoes 전체 original_audio 개수 확인
print("전체 Echoes original_audio :", echoes["original_audio"].nunique())

print(
    "TTA original_audio         :",
    echoes.loc[echoes["type"] == "TTA", "original_audio"].nunique(),
)

print(
    "ATA original_audio         :",
    echoes.loc[echoes["type"] == "ATA", "original_audio"].nunique(),
)

print("Clean TTA original_audio   :", tta_clean["original_audio"].nunique())

전체 Echoes original_audio : 296
TTA original_audio         : 296
ATA original_audio         : 296
Clean TTA original_audio   : 296


## 11. 생성기 내부의 반복 생성 여부 확인

Echoes에서는 하나의 `original_audio`와 연결된 AI 생성 음악이 여러 개 존재할 수 있다.

이를 확인하기 위해 전체 TTA 생성물이 각각 300개인 Suno, Udio, ElevenLabs에 대해 다음 정보를 조사하였다.

- 전체 TTA 생성물 수
- 고유 `original_audio` 수
- 두 개 이상의 생성 결과를 가진 `original_audio` 수

확인 결과 Suno와 Udio는 각각 151개의 고유 reference를 이용하여 대부분 reference당 약 2개의 음악을 생성하였다.

ElevenLabs는 150개의 고유 reference 각각에 대해 정확히 2개의 음악을 생성하여 총 300개의 TTA sample을 구성하였다.

따라서 동일한 `original_audio`가 여러 번 등장하는 것은 단순한 중복 오류가 아니라, **하나의 reference를 바탕으로 서로 다른 AI 생성 결과가 여러 개 만들어진 데이터 구조**임을 확인하였다.

In [28]:
# 생성기 내부의 반복 생성 여부 확인
for gen in ["suno", "udio", "elevenlabs"]:
    temp = tta_clean[tta_clean["generator"] == gen]

    counts = temp["original_audio"].value_counts()
    duplicates = counts[counts > 1]

    print(f"\n===== {gen.upper()} =====")
    print("전체 TTA:", len(temp))
    print("고유 original_audio:", temp["original_audio"].nunique())
    print("중복 original_audio 종류:", len(duplicates))

    print("\n2개 이상 존재하는 original_audio:")
    print(duplicates)


===== SUNO =====
전체 TTA: 300
고유 original_audio: 151
중복 original_audio 종류: 149

2개 이상 존재하는 original_audio:
original_audio
Acoustic Unleashed - Remain                    2
Ad Astra - P C III                             2
All of Us - Eric Skiff                         2
Aquamarine, My Distant Blue - Nihilore         2
Autobahn - S-B-J                               2
                                              ..
Who Loves You Dear - Mink Lungs                2
Windows of the Skull - Sarin                   2
Wir Werden Gott - Japanische Kampfhorspiele    2
Ynkelig - Die Morgendammerung Des Valhalla     2
Новый Нью-Йорк 2 - Чокнутый Пропеллер          2
Name: count, Length: 149, dtype: int64

===== UDIO =====
전체 TTA: 300
고유 original_audio: 151
중복 original_audio 종류: 149

2개 이상 존재하는 original_audio:
original_audio
Acoustic Unleashed - Remain                    2
Ad Astra - P C III                             2
All of Us - Eric Skiff                         2
Aquamarine, My Distant Blue - N

## 12. 생성기별 데이터 생성 구조 비교

Echoes의 12개 AI 생성기가 동일한 방식으로 데이터를 구성하고 있는지 확인하기 위해 생성기별 통계를 계산하였다.

각 생성기에 대해 다음 값을 확인하였다.

- `total_tta`: 전체 TTA 생성물 수
- `unique_original_audio`: 사용된 고유 reference 수
- `min_outputs_per_original`: reference 하나당 최소 생성 결과 수
- `max_outputs_per_original`: reference 하나당 최대 생성 결과 수
- `mean_outputs_per_original`: reference 하나당 평균 생성 결과 수

분석 결과 생성기마다 데이터 구성 방식에 차이가 존재하였다.

예를 들어 AudioLDM과 MusicGen은 대부분 하나의 reference에서 하나의 음악을 생성하는 반면, Suno, Udio, ElevenLabs, Brev는 상대적으로 적은 수의 reference를 사용하면서 reference당 약 2개의 음악을 생성하였다.

따라서 Echoes 데이터는 생성기마다 사용하는 reference의 범위와 생성 횟수가 동일하지 않으며, 이러한 차이는 이후 generator별 성능 비교와 Unseen Generator 실험을 해석할 때 고려해야 한다.

In [29]:
# 생성기별 데이터 생성 구조 비교
rows = []

for gen in sorted(tta_clean["generator"].unique()):

    temp = tta_clean[tta_clean["generator"] == gen]

    counts = temp["original_audio"].value_counts()

    rows.append(
        {
            "generator": gen,
            "total_tta": len(temp),
            "unique_original_audio": temp["original_audio"].nunique(),
            "min_outputs_per_original": counts.min(),
            "max_outputs_per_original": counts.max(),
            "mean_outputs_per_original": counts.mean(),
        }
    )

generator_summary = pd.DataFrame(rows)

display(generator_summary)

,generator,total_tta,unique_original_audio,min_outputs_per_original,max_outputs_per_original,mean_outputs_per_original
0,acestep,294,293,1,2,1.003413
1,audioldm,292,292,1,1,1.000000
2,brev,298,150,1,2,1.986667
3,diffrhythm,299,289,1,3,1.034602
4,elevenlabs,300,150,2,2,2.000000
5,mubert,149,148,1,2,1.006757
6,musicgen,293,293,1,1,1.000000
7,producer,151,150,1,2,1.006667
8,songgen,292,290,1,3,1.006897
9,stableaudio,194,185,1,3,1.048649


## 13. Suno와 ElevenLabs의 reference 집합 비교

각 생성기가 전체 296개의 `original_audio`를 모두 사용하는지 확인하기 위해 Suno와 ElevenLabs가 사용하는 reference 집합을 비교하였다.

확인한 값은 다음과 같다.

- Suno reference 수
- ElevenLabs reference 수
- 두 생성기가 공통으로 사용하는 reference 수
- 두 생성기 reference의 합집합 크기

확인 결과 Suno는 151개의 reference를 사용하였고, ElevenLabs는 150개의 reference를 사용하였다.

두 생성기가 공통으로 사용하는 reference는 150개였으며, 합집합은 151개였다.

즉 ElevenLabs가 사용하는 150개의 reference가 거의 모두 Suno의 reference 집합에 포함되어 있으며, Suno가 추가로 1개의 reference를 사용하고 있음을 알 수 있다.

따라서 모든 생성기가 전체 296개의 reference를 동일하게 사용하는 것이 아니라, **generator마다 서로 다른 reference subset을 사용한다.**

In [30]:
# Suno와 ElevenLabs의 reference 집합 비교
suno_set = set(tta_clean.loc[tta_clean["generator"] == "suno", "original_audio"])

eleven_set = set(
    tta_clean.loc[tta_clean["generator"] == "elevenlabs", "original_audio"]
)

print("Suno reference:", len(suno_set))
print("ElevenLabs reference:", len(eleven_set))
print("공통 reference:", len(suno_set & eleven_set))
print("두 생성기의 union:", len(suno_set | eleven_set))

Suno reference: 151
ElevenLabs reference: 150
공통 reference: 150
두 생성기의 union: 151


In [ ]:
# Suno와 ElevenLabs의 reference 집합 비교
for gen in ["suno", "udio", "elevenlabs"]:
    temp = tta_clean[tta_clean["generator"] == gen]

    counts = temp["original_audio"].value_counts()
    duplicates = counts[counts > 1]

    print(f"\n===== {gen.upper()} =====")
    print("전체 TTA:", len(temp))
    print("고유 original_audio:", temp["original_audio"].nunique())
    print("중복 original_audio 종류:", len(duplicates))

    print("\n2개 이상 존재하는 original_audio:")
    print(duplicates)


===== SUNO =====
전체 TTA: 300
고유 original_audio: 151
중복 original_audio 종류: 149

2개 이상 존재하는 original_audio:
original_audio
Acoustic Unleashed - Remain                    2
Ad Astra - P C III                             2
All of Us - Eric Skiff                         2
Aquamarine, My Distant Blue - Nihilore         2
Autobahn - S-B-J                               2
                                              ..
Who Loves You Dear - Mink Lungs                2
Windows of the Skull - Sarin                   2
Wir Werden Gott - Japanische Kampfhorspiele    2
Ynkelig - Die Morgendammerung Des Valhalla     2
Новый Нью-Йорк 2 - Чокнутый Пропеллер          2
Name: count, Length: 149, dtype: int64

===== UDIO =====
전체 TTA: 300
고유 original_audio: 151
중복 original_audio 종류: 149

2개 이상 존재하는 original_audio:
original_audio
Acoustic Unleashed - Remain                    2
Ad Astra - P C III                             2
All of Us - Eric Skiff                         2
Aquamarine, My Distant Blue - N

## (요약)생성기별 reference 구조 정리

현재 Echoes Clean TTA 데이터에는 총 **296개의 고유 `original_audio`**가 존재하지만, 각 생성기가 이 296개를 모두 사용하는 것은 아니다.

생성기별 고유 reference 수는 서로 다르며, 하나의 reference에서 생성하는 AI 음악의 개수 역시 생성기마다 차이가 있다.

예를 들어 ACE-Step, AudioLDM, MusicGen, SongGen 등은 296개에 가까운 reference를 사용하면서 대부분 reference당 하나의 결과를 생성한다.

반면 Suno, Udio, ElevenLabs, Brev 등은 약 150개의 reference를 이용하면서 하나의 reference에서 약 두 개의 결과를 생성하는 구조를 가진다.

따라서 최종 Clean TTA **3,162개**는 296개의 reference와 12개의 generator 조합에서 동일한 방식으로 만들어진 데이터가 아니라, 생성기별로 서로 다른 reference 범위와 생성 횟수를 가진 데이터로 구성되어 있다.

## (요약)original_audio 기준 Group Split의 필요성

하나의 `original_audio`에서 여러 생성기의 AI 음악 또는 동일 생성기의 여러 생성 결과가 만들어질 수 있다.

따라서 3,162개의 FAKE 파일을 개별 파일 단위로 무작위 분할할 경우, 동일한 reference에 연결된 음악이 Train과 Test에 동시에 포함될 가능성이 있다.

예를 들어 하나의 reference에 대해 생성된 Suno 결과 중 일부가 Train에 들어가고 다른 결과가 Test에 들어간다면, 모델이 완전히 새로운 source family를 평가받는다고 보기 어렵다.

이러한 데이터 누수를 방지하기 위해 이후 Train / Validation / Test 분할은 개별 AI 음악 파일이 아니라 **`original_audio`를 하나의 그룹으로 묶어 수행한다.**

즉 동일한 `original_audio`에 연결된 모든 REAL 및 FAKE 데이터는 반드시 동일한 split에 포함되도록 구성한다.

## FMA 복수 후보 확인

296개 원곡 중 279개는 후보가 한 곡이고 17개는 복수다. 아래에서 라이선스와 장르를 확인해 최종 FMA track ID를 정한다.

## 14. 복수 후보 선택 규칙

296개 `original_audio`를 FMA metadata와 매칭한 결과,
279개는 하나의 FMA track과 일치했지만 17개는 동일한 제목과
아티스트명을 가진 FMA track이 두 개 이상 존재하였다.

Echoes 공개 manifest에는 원본 FMA의 `track_id`가 포함되어 있지 않기 때문에,
복수 후보에서 원 연구자가 사용한 정확한 track ID를 manifest만으로
직접 복원할 수 없다.

따라서 최종 REAL track을 선택하기 위해 다음과 같은 재현 가능한
규칙을 적용한다.

1. Echoes 논문의 bona-fide 데이터 선정 조건인
   CC0, CC-BY 또는 Public Domain 라이선스 후보를 우선한다.
2. 후보가 여러 개 남는 경우 Echoes의 `genre`와
   FMA의 `genre_top`이 일치하는 후보를 우선한다.
3. 동일 조건의 후보가 여러 개 남는 경우 가장 낮은 `track_id`를 선택한다.
4. 허용 라이선스를 만족하는 후보가 없는 경우에는
   genre가 일치하는 후보를 우선하고,
   이후 가장 낮은 `track_id`를 고정된 tie-breaker로 사용한다.

이 규칙은 복수 후보를 임의로 선택하지 않고,
동일한 코드를 다시 실행했을 때 항상 동일한 결과가 나오도록 하기 위한 것이다.

In [33]:
def is_echoes_license(license_value):
    """
    Echoes의 bona-fide 선정 조건에 맞는 라이선스 여부 확인
    """
    if pd.isna(license_value):
        return False

    s = str(license_value).lower()

    # CC0 / Public Domain
    if "cc0" in s or "public domain" in s:
        return True

    # CC-BY 계열
    if "attribution" in s:
        excluded = [
            "noncommercial",
            "non-commercial",
            "no derivatives",
            "noderivatives",
            "sharealike",
            "share alike",
        ]

        if not any(x in s for x in excluded):
            return True

    return False


# --------------------------------------------------
# 모든 FMA 후보 생성
# --------------------------------------------------

fma_candidates = originals[["original_audio", "genre", "match_key"]].merge(
    fma_simple, on="match_key", how="left"
)

# 라이선스 조건
fma_candidates["license_allowed"] = fma_candidates["license"].apply(is_echoes_license)

# 장르 일치 여부
fma_candidates["genre_match"] = (
    fma_candidates["genre"].astype(str).str.lower()
    == fma_candidates["genre_top"].astype(str).str.lower()
)


# --------------------------------------------------
# original_audio별 최종 후보 선택
# --------------------------------------------------

selected_rows = []

for original_audio, group in fma_candidates.groupby("original_audio"):

    group = group.copy()

    # 후보 개수 기록
    candidate_count = len(group)

    # 1. 라이선스 조건을 만족하는 후보 우선
    allowed = group[group["license_allowed"]]

    if len(allowed) > 0:
        pool = allowed.copy()
        license_fallback = False
    else:
        pool = group.copy()
        license_fallback = True

    # 2. Echoes genre와 FMA genre_top이 일치하는 후보 우선
    genre_matched = pool[pool["genre_match"]]

    if len(genre_matched) > 0:
        pool = genre_matched.copy()

    # 3. 그래도 여러 개면 가장 낮은 track_id 선택
    selected = pool.sort_values("track_id").iloc[0].copy()

    # 그룹 정보를 명시적으로 다시 저장
    selected["original_audio"] = original_audio
    selected["candidate_count"] = candidate_count
    selected["license_fallback"] = license_fallback

    selected_rows.append(selected)


# 최종 DataFrame 생성
selected_real = pd.DataFrame(selected_rows).reset_index(drop=True)


# --------------------------------------------------
# 결과 확인
# --------------------------------------------------

print("===== FINAL FMA REAL MAPPING =====")
print("Original audio :", selected_real["original_audio"].nunique())
print("Selected rows  :", len(selected_real))
print("Selected tracks:", selected_real["track_id"].nunique())

print("\n===== CANDIDATE COUNTS =====")
print(selected_real["candidate_count"].value_counts().sort_index())

print("\n===== LICENSE FALLBACK =====")
print(selected_real["license_fallback"].value_counts())

print("\n===== GENRE MATCH =====")
print(selected_real["genre_match"].value_counts(dropna=False))

===== FINAL FMA REAL MAPPING =====
Original audio : 296
Selected rows  : 296
Selected tracks: 296

===== CANDIDATE COUNTS =====
candidate_count
1    279
2     16
5      1
Name: count, dtype: int64

===== LICENSE FALLBACK =====
license_fallback
False    265
True      31
Name: count, dtype: int64

===== GENRE MATCH =====
genre_match
True    296
Name: count, dtype: int64


## 14-1. 31개가 어떤 곡인지 확인

In [34]:
# 31개가 어떤 곡인지 확인
multiple_selected = selected_real[selected_real["candidate_count"] > 1].copy()

display(
    multiple_selected[
        [
            "original_audio",
            "genre",
            "track_id",
            "genre_top",
            "license",
            "subset",
            "candidate_count",
            "license_allowed",
            "license_fallback",
            "genre_match",
        ]
    ]
)

,original_audio,genre,track_id,genre_top,license,subset,candidate_count,license_allowed,license_fallback,genre_match
1,1984 - Punk Rock Opera,Rock,149410,Rock,Attribution,medium,2,True,False,True
18,"Aquamarine, My Distant Blue - Nihilore",Electronic,134150,Electronic,Creative Commons Attribution,large,2,True,False,True
20,As Nihilism Gives Way To Existentialism - Nihi...,Electronic,134148,Electronic,Creative Commons Attribution,large,2,True,False,True
96,I Know His Blood - Vienna Ditto,Electronic,107616,Electronic,Attribution,small,2,True,False,True
98,I'm gonna try to reach - Los Llamarada,Rock,28763,Rock,Attribution-NoDerivatives 3.0 International,large,2,False,True,True
110,KOMFORT - voyageurs,Rock,42772,Rock,Attribution 3.0 International,medium,2,True,False,True
122,Let You're Body Move - D SMILEZ,Electronic,136017,Electronic,Attribution,large,2,True,False,True
130,Lost In The Music (D-Smilez Mix) - D SMILEZ,Electronic,136016,Electronic,Attribution,large,2,True,False,True
133,Loved Ones - Rowan Box,Electronic,148786,Electronic,Attribution,large,2,True,False,True
140,Monkeystage - Ergo Phizmiz,Pop,22477,Pop,Attribution 3.0 United States,small,2,True,False,True


## 14-2. 논문에서는 각 bona-fide track의 title과 genre를 이용해서 description을 만들었다고 설명(판별 코드)

In [35]:
# 논문에서는 각 bona-fide track의 title과 genre를 이용해서 description을 만들었다고 설명(판별 코드)
reference_check = (
    tta_clean.groupby("original_audio")
    .agg(
        total_fake=("path_in_dataset", "size"),
        generator_count=("generator", "nunique"),
        description_count=("description", "nunique"),
        genre_count=("genre", "nunique"),
    )
    .reset_index()
)

print("전체 original_audio:", len(reference_check))

print("\n===== description이 2개 이상인 original_audio =====")

multi_description = reference_check[
    reference_check["description_count"] > 1
].sort_values(["description_count", "original_audio"], ascending=[False, True])

print("개수:", len(multi_description))

display(multi_description)

전체 original_audio: 296

===== description이 2개 이상인 original_audio =====
개수: 2


,original_audio,total_fake,generator_count,description_count,genre_count
69,"First Glance - Oh Yeah, the Future",14,10,2,1
163,OST 04 Ship under attack - sawsquarenoise,16,12,2,1


## 14-3. 17개 복수 후보와 license fallback이 얼마나 겹쳤는지도 확인

In [36]:
# 17개 복수 후보와 license fallback이 얼마나 겹쳤는지도 확인
print("===== Multiple candidate + License fallback =====")

display(
    selected_real[
        (selected_real["candidate_count"] > 1) | (selected_real["license_fallback"])
    ][
        [
            "original_audio",
            "genre",
            "track_id",
            "license",
            "candidate_count",
            "license_allowed",
            "license_fallback",
            "genre_match",
        ]
    ].sort_values(
        ["license_fallback", "candidate_count"], ascending=[False, False]
    )
)

===== Multiple candidate + License fallback =====


,original_audio,genre,track_id,license,candidate_count,license_allowed,license_fallback,genre_match
98,I'm gonna try to reach - Los Llamarada,Rock,28763,Attribution-NoDerivatives 3.0 International,2,False,True,True
220,Take Your Fingers - Michael Fakesch,Electronic,33594,Attribution-NoDerivatives 3.0 International,2,False,True,True
17,Apparitions Under Glass - Sarin,Rock,113526,Attribution-NoDerivatives 4.0 International,1,False,True,True
40,Coliidae - K.D. Expression,Electronic,33636,Attribution-NoDerivatives 3.0 International,1,False,True,True
44,Crossing - Los Llamarada,Rock,28753,Attribution-NoDerivatives 3.0 International,1,False,True,True
46,Dance of the Martians (SynthStep Edit) - Maxim...,Electronic,136338,Attribution-NoDerivatives 4.0 International,1,False,True,True
47,Daylight Savings - My brother Daniel,Electronic,117280,Attribution-NoDerivatives 4.0 International,1,False,True,True
68,Fifth Ramble - From the album Keith [RSVP007] ...,Electronic,60834,Attribution-NoDerivatives 3.0 International,1,False,True,True
78,Fridge - Railkid Station,Rock,94342,Attribution-NoDerivatives 3.0 International,1,False,True,True
87,"Head of Ancante, Talking Tree - Ak'chamel, The...",Rock,135209,Attribution-NoDerivatives 4.0 International,1,False,True,True


## 14-4. 복수 Description 원곡 상세 확인

296개의 `original_audio` 중 2개의 원곡에서 서로 다른 두 종류의
description이 확인되었다.

두 경우 모두 genre는 하나로 일관되어 있으므로 장르 metadata가
혼합된 문제는 아니다.

따라서 해당 원곡에 연결된 생성기와 description 내용을 직접 확인하여
단순한 prompt 표현 차이인지, 서로 다른 source 정보가 혼합된 것인지
추가로 점검한다.

In [37]:
# 복수 Description 원곡 상세 확인
multi_desc_names = multi_description["original_audio"].tolist()

multi_desc_detail = tta_clean[tta_clean["original_audio"].isin(multi_desc_names)][
    ["original_audio", "generator", "genre", "description", "path_in_dataset"]
].sort_values(["original_audio", "description", "generator"])

display(multi_desc_detail)

,original_audio,generator,genre,description,path_in_dataset
69,"First Glance - Oh Yeah, the Future",acestep,Pop,No description available,TTA/acestep/First_Glance_Oh_Yeah_the_Future_ac...
995,"First Glance - Oh Yeah, the Future",brev,Pop,No description available,TTA/brev/First_Glance_Oh_Yeah_the_Future_brev_...
996,"First Glance - Oh Yeah, the Future",brev,Pop,No description available,TTA/brev/First_Glance_Oh_Yeah_the_Future_brev_...
1490,"First Glance - Oh Yeah, the Future",diffrhythm,Pop,No description available,TTA/diffrhythm/First_Glance_Oh_Yeah_the_Future...
4015,"First Glance - Oh Yeah, the Future",musicgen,Pop,No description available,TTA/musicgen/First_Glance_Oh_Yeah_the_Future_m...
2039,"First Glance - Oh Yeah, the Future",producer,Pop,No description available,TTA/producer/First_Glance_Oh_Yeah_the_Future_p...
2348,"First Glance - Oh Yeah, the Future",songgen,Pop,No description available,TTA/songgen/First_Glance_Oh_Yeah_the_Future_so...
3438,"First Glance - Oh Yeah, the Future",stableaudio,Pop,No description available,TTA/stableaudio/First_Glance_Oh_Yeah_the_Futur...
2899,"First Glance - Oh Yeah, the Future",suno,Pop,No description available,TTA/suno/First_Glance_Oh_Yeah_the_Future_suno_...
2900,"First Glance - Oh Yeah, the Future",suno,Pop,No description available,TTA/suno/First_Glance_Oh_Yeah_the_Future_suno_...


## 복수 Description 문자열 상세 검증

296개의 `original_audio` 중 두 원곡에서 `description_count`가 2로 확인되었다.

그러나 단순히 description의 개수가 2라는 사실만으로 서로 다른 원곡 정보가
혼합되었다고 판단할 수는 없다.

예를 들어 일부 생성기에서는 실제 description 대신
`No description available`이 기록되어 있을 수 있으며,
공백이나 미세한 문자열 차이로 인해 서로 다른 description으로 집계될 수도 있다.

따라서 두 원곡에 대해 고유 description 문자열을 원문 그대로 출력하고,
각 description을 사용한 generator를 확인한다.

In [38]:
# 복수 Description 문자열 상세 검증
for name in multi_desc_names:
    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)

    temp = tta_clean[tta_clean["original_audio"] == name]

    descriptions = temp["description"].drop_duplicates()

    print("Unique descriptions:", len(descriptions))

    for i, desc in enumerate(descriptions, start=1):
        print(f"\n[Description {i}]")
        print(repr(desc))

        gens = temp.loc[temp["description"] == desc, "generator"].value_counts()

        print("\nGenerators:")
        print(gens)


First Glance - Oh Yeah, the Future
Unique descriptions: 2

[Description 1]
'No description available'

Generators:
generator
brev           2
suno           2
udio           2
acestep        1
diffrhythm     1
producer       1
songgen        1
stableaudio    1
musicgen       1
Name: count, dtype: int64

[Description 2]
'indie-synthpop, dreamy-pads, airy-male-vocal, midtempo, nostalgic, melodic, reverb, gentle, romantic, hazy'

Generators:
generator
elevenlabs    2
Name: count, dtype: int64

OST 04 Ship under attack - sawsquarenoise
Unique descriptions: 2

[Description 1]
'chiptune, frantic-tempo, staccato-arps, alarm-like, tense, energetic, 8-bit, boss-fight, looping, instrumental'

Generators:
generator
brev           2
suno           2
udio           2
acestep        1
audioldm       1
diffrhythm     1
mubert         1
producer       1
songgen        1
stableaudio    1
musicgen       1
Name: count, dtype: int64

[Description 2]
'chiptune, frantic-tempo, staccato-arps, alarm-like, te

## 15. FMA REAL 매핑 최종 확정

Echoes의 296개 `original_audio`와 FMA metadata를 매칭하고 추가적인 품질 검증을 수행하였다.

확인 결과:

* 전체 `original_audio`: **296개**
* FMA 단일 후보: **279개**
* FMA 복수 후보: **17개**
* 미매칭: **0개**
* 최종 선택된 FMA track: **296개**
* Echoes genre와 FMA `genre_top` 일치: **296 / 296**

또한 일부 원곡에서 서로 다른 description이 확인되었지만,
이는 서로 다른 source track이 혼합된 것이 아니라 생성기별 prompt 또는
metadata 기록 방식의 차이에 의한 것으로 판단하였다.

따라서 현재 선택된 296개의 FMA `track_id`를 REAL 음악 확보를 위한
최종 mapping으로 사용한다.

라이선스 관련 정보와 후보 수 등의 품질 검증 정보는 이후 데이터 재현성과
검토를 위해 mapping 파일에 함께 보존한다.

In [39]:
from pathlib import Path

output_dir = PROJECT_ROOT / "data/metadata"
output_dir.mkdir(parents=True, exist_ok=True)

mapping_path = output_dir / "fma_real_mapping.csv"

mapping_columns = [
    "original_audio",
    "genre",
    "track_id",
    "title",
    "artist",
    "genre_top",
    "license",
    "duration",
    "subset",
    "candidate_count",
    "license_allowed",
    "license_fallback",
    "genre_match",
]

fma_real_mapping = (
    selected_real[mapping_columns].sort_values("original_audio").reset_index(drop=True)
)

fma_real_mapping.to_csv(mapping_path, index=False, encoding="utf-8-sig")

print("Saved:", mapping_path)
print("Rows:", len(fma_real_mapping))
print("Unique original_audio:", fma_real_mapping["original_audio"].nunique())
print("Unique track_id:", fma_real_mapping["track_id"].nunique())

display(fma_real_mapping.head(10))

Saved: /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/metadata/fma_real_mapping.csv
Rows: 296
Unique original_audio: 296
Unique track_id: 296


,original_audio,genre,track_id,title,artist,genre_top,license,duration,subset,candidate_count,license_allowed,license_fallback,genre_match
0,"10,000 People Chanting, ""I'm an Individual"" - ...",Electronic,140932,"10,000 People Chanting, ""I'm an Individual""",Nihilore,Electronic,Creative Commons Attribution,372,medium,1,True,False,True
1,1984 - Punk Rock Opera,Rock,149410,1984,Punk Rock Opera,Rock,Attribution,200,medium,2,True,False,True
2,2 (Wasn't There) - Isle of Pine,Rock,66449,2 (Wasn't There),Isle of Pine,Rock,Attribution-NoDerivs 2.5 Canada,108,medium,1,True,False,True
3,2Much (Andy Spinelli & Alex Sánchez House Edit...,Electronic,114244,2Much (Andy Spinelli & Alex Sánchez House Edit),Tentacles,Electronic,Attribution,486,medium,1,True,False,True
4,3 am West End - statusq,Electronic,112378,3 am West End,statusq,Electronic,Attribution,291,medium,1,True,False,True
5,5 (Lexington) - Isle of Pine,Rock,66445,5 (Lexington),Isle of Pine,Rock,Attribution-NoDerivs 2.5 Canada,157,large,1,True,False,True
6,"50,000 Volts of Democracy mp3 - Legally Blind",Rock,130401,"50,000 Volts of Democracy mp3",Legally Blind,Rock,Attribution,270,medium,1,True,False,True
7,"6 (Coat of Arms, Close) - Isle of Pine",Rock,66446,"6 (Coat of Arms, Close)",Isle of Pine,Rock,Attribution-NoDerivs 2.5 Canada,204,large,1,True,False,True
8,A Dark Blue Arc - Pipe Choir,Rock,129963,A Dark Blue Arc,Pipe Choir,Rock,Attribution,327,medium,1,True,False,True
9,A Different World By Night - Nihilore,Electronic,140926,A Different World By Night,Nihilore,Electronic,Creative Commons Attribution,296,small,1,True,False,True


## 16. 최종 REAL Track의 FMA Subset 분포 확인

최종 매핑된 296개의 FMA REAL track이 `small`, `medium`, `large`
중 어느 subset에 포함되어 있는지 확인한다.

FMA 오디오 전체를 다운로드하기 전에 subset 분포를 확인하여,
필요한 REAL track을 확보하기 위한 최소 다운로드 범위를 결정한다.

In [40]:
# 최종 REAL Track의 FMA Subset 분포 확인
print("===== FMA SUBSET DISTRIBUTION =====")

print(selected_real["subset"].value_counts(dropna=False))

print("\nTotal:", len(selected_real))

===== FMA SUBSET DISTRIBUTION =====
subset
small     122
medium    103
large      71
Name: count, dtype: int64

Total: 296


In [41]:
# 최종 REAL Track의 FMA Subset 분포 확인
subset_genre = pd.crosstab(selected_real["subset"], selected_real["genre"])

display(subset_genre)

genre,Electronic,Pop,Rock
subset,,,
large,26,11,34
medium,47,0,56
small,45,56,21


## 16. FMA REAL 음원 개별 확보 가능 여부 확인

최종 REAL 매핑 결과는 small 122곡, medium 103곡, large 71곡으로 구성된다.

FMA large 전체 데이터는 약 93GB이므로, 296개의 REAL 음악만 사용하기 위해
전체 archive를 다운로드하는 것은 비효율적이다.

FMA의 원본 데이터 생성 과정에서는 `raw_tracks.csv`의 `track_file` 정보를 이용해
각 음악 파일을 개별적으로 다운로드한 뒤, 30초가 넘는 음악의 중앙 30초 구간을
추출하여 `fma_large` clip을 생성하였다.

따라서 먼저 현재 `raw_tracks.csv`에 필요한 296곡의 `track_file` 정보가 존재하는지
확인하고, 개별 음원 다운로드가 가능한지 점검한다.

In [42]:
RAW_TRACKS = PROJECT_ROOT / "data/raw/FMA/fma_metadata/raw_tracks.csv"

raw_tracks = pd.read_csv(RAW_TRACKS, index_col=0)

print("raw_tracks shape:", raw_tracks.shape)

print("\n===== 관련 컬럼 =====")
print(
    [
        col
        for col in raw_tracks.columns
        if "file" in col.lower() or "duration" in col.lower()
    ]
)

raw_tracks shape: (109727, 38)

===== 관련 컬럼 =====
['license_image_file', 'license_image_file_large', 'track_duration', 'track_file', 'track_image_file']


In [43]:
# FMA REAL 음원 개별 확보 가능 여부 확인
selected_ids = selected_real["track_id"].astype(int).tolist()

raw_selected = raw_tracks.loc[raw_tracks.index.intersection(selected_ids)].copy()

print("필요한 track_id:", len(selected_ids))
print("raw_tracks에서 찾은 track_id:", len(raw_selected))

display(raw_selected[["track_file", "track_duration"]].head(10))

필요한 track_id: 296
raw_tracks에서 찾은 track_id: 296


,track_file,track_duration
track_id,,
1382,music/WFMU/Parsley_Flakes/Parsley_Flakes_mp3s/...,02:10
1881,music/WFMU/The_Tleilaxu_Music_Machine/The_Tlei...,04:13
3836,music/WFMU/Lightning_Bolt/Live_at_WFMU_on_Bria...,05:01
3857,music/WFMU/Los_Fancy_Free/Live_at_WFMU_on_Liz_...,03:17
3936,music/WFMU/Mink_Lungs/Live_at_WFMU_on_Scotts_S...,02:10
3956,music/WFMU/Miss_Derringer/Live_at_WFMU_of_Joe_...,02:38
3959,music/WFMU/Miss_Derringer/Live_at_WFMU_of_Joe_...,02:47
3961,music/WFMU/Mod_Fun/Live_at_WFMU_on_Pat_Duncans...,04:24
3962,music/WFMU/Mod_Fun/Live_at_WFMU_on_Pat_Duncans...,03:52


In [44]:
# FMA REAL 음원 개별 확보 가능 여부 확인
raw_selected["download_url"] = "https://files.freemusicarchive.org/" + raw_selected[
    "track_file"
].astype(str)

display(raw_selected[["track_file", "track_duration", "download_url"]].head(5))

,track_file,track_duration,download_url
track_id,,,
1382,music/WFMU/Parsley_Flakes/Parsley_Flakes_mp3s/...,02:10,https://files.freemusicarchive.org/music/WFMU/...
1881,music/WFMU/The_Tleilaxu_Music_Machine/The_Tlei...,04:13,https://files.freemusicarchive.org/music/WFMU/...
3836,music/WFMU/Lightning_Bolt/Live_at_WFMU_on_Bria...,05:01,https://files.freemusicarchive.org/music/WFMU/...
3857,music/WFMU/Los_Fancy_Free/Live_at_WFMU_on_Liz_...,03:17,https://files.freemusicarchive.org/music/WFMU/...
3936,music/WFMU/Mink_Lungs/Live_at_WFMU_on_Scotts_S...,02:10,https://files.freemusicarchive.org/music/WFMU/...


In [45]:
import requests

HF_DATASET = "benjamin-paine/free-music-archive-large"
HF_API = "https://datasets-server.huggingface.co/filter"

params = {
    "dataset": HF_DATASET,
    "config": "default",
    "split": "train",
    "where": "\"title\"='1984' AND \"artist\"='Punk Rock Opera'",
    "length": 10,
}

response = requests.get(HF_API, params=params, timeout=60)

print("status:", response.status_code)
print("URL:", response.url)

data = response.json()

print("검색 결과 수:", len(data.get("rows", [])))

for result in data.get("rows", []):
    row = result["row"]

    print("\nTitle :", row.get("title"))
    print("Artist:", row.get("artist"))
    print("Audio :", row.get("audio"))

status: 500
URL: https://datasets-server.huggingface.co/filter?dataset=benjamin-paine%2Ffree-music-archive-large&config=default&split=train&where=%22title%22%3D%271984%27+AND+%22artist%22%3D%27Punk+Rock+Opera%27&length=10
검색 결과 수: 0


## FMA 매칭 결과

- 296개 `original_audio` 중 단일 후보 279개, 복수 후보 17개, 미매칭 0개다.
- 라이선스 → 장르 → 최소 `track_id` 순의 재현 가능한 규칙으로 **296개 고유 FMA track**을 확정했다.
- Echoes 장르와 최종 FMA `genre_top`은 **296/296 일치**한다.
- FMA subset 분포는 small 122곡, medium 103곡, large 71곡이다.
- `data/metadata/fma_real_mapping.csv` 저장을 완료했다.